# Part 9 · Notebook 02 — Cointegration and screening for pairs

**Sessions:** S2 (Cointegration & pair selection) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. See why correlated returns don't make a pair.
2. Run the Engle–Granger test and recover a known hedge ratio.
3. Control false discoveries across many candidate pairs.
4. Screen a universe within sectors, and see what screening everything costs.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. Correlation is not cointegration

Two stocks can move together day by day (correlated **returns**) and still drift apart forever. A pair trade needs the **prices** to share a stationary spread: cointegration.

In [ ]:
rng = np.random.default_rng(5)
common = np.cumsum(rng.normal(0, 0.01, 1500))
a = pd.Series(4 + common + np.cumsum(rng.normal(0, 0.006, 1500)))
b = pd.Series(4 + common + np.cumsum(rng.normal(0, 0.006, 1500)))
pair = p.cointegrated_pair()                          # y = 0.5 + 1.5·x + an OU spread with half-life 7
print(f"correlated random walks: return correlation {np.corrcoef(np.diff(a), np.diff(b))[0, 1]:.2f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(a.to_numpy() - b.to_numpy()); axes[0].set_title("spread of two correlated random walks: wanders off")
axes[1].plot((pair.y - 1.5 * pair.x).to_numpy()); axes[1].set_title("spread of a cointegrated pair: comes back")
plt.tight_layout(); plt.show()

## 2. Engle–Granger

Regress `y = α + β·x` by OLS (`sm.OLS(y, sm.add_constant(x)).fit()`, whose `params` are `[α, β]`), form the spread `y − β·x − α`, and test it for a unit root with the cointegration critical values (`statsmodels`' `coint(y, x)[1]` is the p-value). Add the spread's OU half-life. Return a dict with `alpha`, `beta`, `pvalue`, `spread`, `half_life`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

def engle_granger(y, x):
    ols = sm.OLS(y, sm.add_constant(x)).fit()
    alpha, beta = float(ols.params.iloc[0]), float(ols.params.iloc[1])
    spread = ...                                  # ✍️
    return {"alpha": alpha, "beta": beta, "pvalue": float(coint(y, x)[1]), "spread": spread,
            "half_life": p.fit_ou(spread.to_numpy())["half_life"]}

mine = [p.attempt(engle_granger, pair.y, pair.x), p.attempt(engle_granger, a, b)]
ref = [p.engle_granger(pair.y, pair.x), p.engle_granger(a, b)]
mine = p.check("engle_granger", [{k: v for k, v in m.items() if k != "spread"} if m is not Ellipsis else m for m in mine],
               [{k: v for k, v in r.items() if k != "spread"} for r in ref])
pd.DataFrame(mine, index=["cointegrated pair (β 1.5, half-life 7)", "correlated random walks"]).round(4)

## 3. Many pairs, many tests

A sector of 8 stocks has 28 pairs; 500 stocks have 124,750. At a 5% threshold, one in twenty unrelated pairs passes by luck. **Benjamini–Hochberg** controls the false discovery rate: sort the p-values, find the **largest** `k` with `p_(k) <= k·q/m`, and reject the `k` smallest. Return a boolean array in the input order.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bh_reject(pvalues, q=0.05):
    pv = np.asarray(pvalues, dtype=float)
    m = pv.size
    order = np.argsort(pv)
    ok = ...                                      # ✍️ sorted p-values <= rank·q/m, rank = 1..m
    out = np.zeros(m, dtype=bool)
    if ok.any():
        k = np.max(np.flatnonzero(ok))
        out[order[:k + 1]] = True
    return out

cases = [[0.001, 0.008, 0.039, 0.041, 0.042, 0.06, 0.074, 0.205, 0.212, 0.216], list(np.random.default_rng(0).uniform(size=50)) + [1e-5]]
mine = [p.attempt(bh_reject, c) for c in cases]
mine = p.check("bh_reject", mine, [p.bh_reject(c) for c in cases])
print(f"case 1: {int(np.sum(mine[0]))} discoveries ({int(np.sum(np.array(cases[0]) < 0.05))} below 0.05);  "
      f"case 2: {int(np.sum(mine[1]))} discovery among 51 tests ({int(np.sum(np.array(cases[1]) < 0.05))} below 0.05)")

## 4. Screen a universe

Four sectors of eight stocks (a market factor, a sector factor and noise); in each sector **one** pair is truly cointegrated. Screen **within sectors** first (economic link), test every pair with Engle–Granger, apply BH-FDR, then keep half-lives between 2 and 30 days (`p.screen_pairs`).

In [ ]:
prices, sectors, truth = p.sector_universe()
screen = p.screen_pairs(prices, sectors)
print("true pairs:", truth)
display(screen.head(8).round(4))
print(f"{len(screen)} pairs tested within sectors: {(screen.pvalue < 0.05).sum()} with raw p < 0.05, {screen.fdr_pass.sum()} pass FDR, "
      f"{screen.selected.sum()} selected")

In [ ]:
everything = p.screen_pairs(prices)               # no economic link: all 496 pairs
found = [tuple(r) for r in everything[everything.fdr_pass][["a", "b"]].to_numpy()]
print(f"{len(everything)} pairs tested across the whole universe: {(everything.pvalue < 0.05).sum()} raw p < 0.05, "
      f"{everything.fdr_pass.sum()} pass FDR; true pairs missed: {[t for t in truth if t not in found]}")

Within sectors, all four true pairs pass and nothing else does. Screening everything quadruples the number of tests, so FDR must be stricter, and one true pair is lost. The economic prefilter isn't just a shortcut: it buys statistical power.

## Wrap-up

* Cointegration of prices, not correlation of returns.
* Economic link first, then Engle–Granger (or Johansen for baskets), then FDR, then a half-life filter.
* Graded version: `labs/part09/week31_pairs` (Johansen too) and Clinic W1 (screen, then trade out of sample).